# 03 - Seasonality, crashes and volatility (Q3, Q5)

Q3: which months and crops crash predictably?  
Q5: how does volatility differ between the perishable (tomato) and storables (onion, potato)?

Crash (spec 9.2): within the next 14 days the price falls below 70% of its trailing 30-day median, computed only from days with data.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import pandas as pd
from mandipulse import viz
from mandipulse.viz import COMMODITIES, COMMODITY_COLORS, INK, INK_2, MUTED, load
pd.set_option("display.width", 160)
viz.style()

## Q3 - seasonal price pattern and crash rate by month

In [ ]:
s = load("phase3", "q3_seasonality_by_month")
fig, ax = viz.figure("Q3. Seasonal price pattern by crop",
                     "Price index = month median / annual median (1.0 = a typical month), median across years and states")
for c in COMMODITIES:
    g = s[s["commodity"] == c]
    ax.plot(g["month"], g["median_price_index"], color=COMMODITY_COLORS[c], marker="o", markersize=5, label=c)
viz.label_line_ends(ax, s, "month", "median_price_index", "commodity", fmt="{:.2f}")
ax.axhline(1, color=MUTED, linewidth=0.8)
ax.set_xticks(range(1, 13), ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])
ax.set_ylabel("price index (x annual median)")
viz.legend(ax)
viz.save(fig, "q3_price_index_by_month")
s.pivot(index="month", columns="commodity", values="median_price_index")

In [ ]:
grid = s.pivot(index="commodity", columns="month", values="crash_rate_pct").loc[COMMODITIES]
fig, ax = viz.figure("Q3. How often prices crash, by month",
                     "% of market-days followed within 14 days by >= 2 days below 70% of the 30-day median", height=3.8)
cmap = LinearSegmentedColormap.from_list("seq", viz.SEQUENTIAL)
im = ax.imshow(grid.values, cmap=cmap, aspect="auto", vmin=0)
ax.grid(False)
ax.set_xticks(range(12), ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])
ax.set_yticks(range(len(grid.index)), grid.index)
vmax = np.nanmax(grid.values)
for i in range(grid.shape[0]):
    for j in range(grid.shape[1]):
        v = grid.values[i, j]
        ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=8.5,
                color="white" if v > 0.55 * vmax else INK)
fig.colorbar(im, ax=ax, label="crash rate, %", shrink=0.8)
viz.save(fig, "q3_crash_rate_heatmap")
grid

In [ ]:
display(load("phase3", "q3_crash_rate_by_state_month").pivot_table(
    index=["commodity", "state"], columns="month", values="crash_rate_pct"))
load("phase3", "q3_crash_lead")

## Q5 - volatility: perishable vs storable

In [ ]:
v = load("phase3", "q5_volatility_by_commodity").set_index("commodity").loc[COMMODITIES].reset_index()
viz.style()
fig, axes = plt.subplots(1, 2, figsize=(10, 4.6))
viz.header(fig, "Q5. Price volatility: tomato vs onion and potato",
           "Median over market-years with >= 60 report days (main period)")
panels = [("median_cv", "coefficient of variation (std / mean)", "{:.2f}"),
          ("median_avg_abs_daily_change_pct", "average absolute day-to-day change, %", "{:.1f}%")]
for ax, (col, label, fmt) in zip(axes, panels, strict=True):
    bars = ax.bar(v["commodity"], v[col], color=[COMMODITY_COLORS[c] for c in v["commodity"]], width=0.55)
    for b, val in zip(bars, v[col], strict=True):
        ax.annotate(fmt.format(val), (b.get_x() + b.get_width() / 2, val), xytext=(0, 3),
                    textcoords="offset points", ha="center", fontsize=9, color=INK_2)
    ax.set_title(label, loc="left", fontsize=10, color=INK_2)
    ax.set_ylim(0)
viz.save(fig, "q5_volatility_by_commodity")
v

In [ ]:
vy = load("phase3", "q5_volatility_by_year")
fig, ax = viz.figure("Q5. Volatility every year", "Median coefficient of variation of daily prices within a market-year")
for c in COMMODITIES:
    g = vy[vy["commodity"] == c]
    ax.plot(g["year"], g["median_cv"], color=COMMODITY_COLORS[c], marker="o", markersize=5, label=c)
viz.label_line_ends(ax, vy, "year", "median_cv", "commodity", fmt="{:.2f}")
ax.set_ylabel("median CV")
ax.set_ylim(0)
viz.legend(ax)
viz.save(fig, "q5_cv_by_year")
vy.pivot(index="year", columns="commodity", values="median_cv")